In [1]:
from os import listdir
from typing import TypeAlias

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.gridspec import GridSpec
from PIL import Image
from skimage.transform import resize

# from dinosaw.models.vit_wrapper import PretrainedViTWrapper, MODEL_LIST
from dinosaw import utils
from dinosaw.linear_probe import (
    LinearProbeResult,
    RampTypes,
    do_linear_probe,
    gen_sample_mask,
    get_ramp,
)
from dinosaw.utils import add_custom_font
from dinosaw.wrappers import (
    ModelTypes,
    PretrainedViTWrapper,
    get_models,
)

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

LOAD_RESULTS = True

/home/ronan/Documents/academic/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
selected_models: tuple[ModelTypes, ...] = ('dinov2_s', 'dinov3_s+', 'alibi_coco_dinov2_s', 'alibi_dinov3_s+')
# selected_models: tuple[ModelTypes, ...] = ('dv_b', 'dv2_b', 'dv3_b', 'clip_b', 'vit_b_in', 'vit_b')

models = get_models(selected_models, DEVICE, True, '../../models/checkpoints',  conf_dir='../../models/dinov3')
S = models[selected_models[0]].stride

n_layers = 12
n_dims = models[selected_models[0]].embed_dim

2026-11-01 14:49:54 | I | factory.py                 : 154 | Building wrapper 'dinov2_s' on device cuda:0
2026-11-01 14:49:54 | I | factory.py                 : 133 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)


2026-11-01 14:49:55 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-11-01 14:49:55 | I | factory.py                 : 154 | Building wrapper 'dinov3_s+' on device cuda:0
2026-11-01 14:49:55 | I | factory.py                 : 133 | Building backbone with config: BackboneConfig(backbone_type='torch_hub', model_arch='dinov3_s+', pretrained=False, checkpoint_path='../../models/checkpoints/backbones/dinov3_vits_patch16_plus_reg4.pth', model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)


/home/ronan/Documents/academic/dino-saw/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU0 NVIDIA GeForce GTX 1080 Ti which is of compute capability (CC) 6.1.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/ronan/Documents/academic/dino-saw/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:489: UserWarning: 
NVIDIA GeForce GTX 1080 Ti with CUDA capability sm_61 is not compatible 

2026-11-01 14:49:55 | I | factory.py                 :  80 | Loading checkpoint: ../../models/checkpoints/backbones/dinov3_vits_patch16_plus_reg4.pth
2026-11-01 14:49:55 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (16, 16), patch_size: (16, 16), embed_dim: 384, num_blocks: 12
2026-11-01 14:49:55 | I | factory.py                 : 154 | Building wrapper 'alibi_coco_dinov2_s' on device cuda:0
2026-11-01 14:49:55 | I | factory.py                 : 133 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=False, checkpoint_path='../../models/checkpoints/trained/alibi_coco_dv2_vits14_reg4.pth', model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=True, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[functools.partial(<function add_alibi at 0x7e7e47ccefc0>, slope_type='constant', n_reg_tokens=4, metric='euclidean'

In [3]:
def get_feature_list(
    model: PretrainedViTWrapper,
    pil_img: Image.Image,
    channel_last: bool = False,
    channel_blank: bool = False,
    N: int = 11
) -> list[np.ndarray]:
    # tr = utils.closest_resize(pil_img.height, pil_img.width, model.stride)
    # img_tensor = utils.convert_image(pil_img, tr, device_str=device, to_half=to_half)

    with torch.no_grad():
        embs = model.forward_intermediates(pil_img, list(range(N)))
    embs_np = [utils.to_numpy(emb.squeeze(0)) for emb in embs]
    if channel_blank:
        channels_to_blank = [47, 113, 117, 359]
        for emb_np in embs_np:
            emb_np[channels_to_blank, :, :] = 0
    if channel_last:
        embs_np = [np.transpose(emb_np, (1, 2, 0)) for emb_np in embs_np]

    return embs_np

In [4]:

if not LOAD_RESULTS:
    ds_folder = 'data/linear_probe/homog_micros'
    image_files = [f for f in listdir(ds_folder)]
    n_imgs = len(image_files)

    features: dict[ModelTypes, list[list[np.ndarray]]] = {key: [] for key in selected_models}

    for key, model in models.items():
        for img_file in image_files:
            img_path = f'{ds_folder}/{img_file}'
            img = Image.open(img_path).convert('RGB')
            feat_list = get_feature_list(model, img, channel_last=True, N=n_layers)
            features[key].append(feat_list)

In [5]:
AverageResult: TypeAlias = tuple[np.ndarray, np.ndarray, float, float, np.ndarray]
def average_results(results: list[LinearProbeResult]) -> AverageResult:
    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [6]:
from pickle import dump as pdump, load as pload
from os import mkdir

DATA_DIR = "data/pos_fingerprint"

In [7]:
# ramps: tuple[RampTypes, ...] = ('lr', 'ud', 'diag', 'radial')
ramp: RampTypes = 'lr'
model_to_layer_results: dict[ModelTypes, list[AverageResult]] = {key: [] for key in selected_models}
MASK_CUTOFF_FRAC = 1
STEP = 6
RANDOM_MASK = True



if LOAD_RESULTS:
    with open(f"{DATA_DIR}/scores.pkl", "rb") as f:
        model_to_layer_channel_scores = pload(f)

else:
    for key, model in models.items():
        for layer in range(n_layers):
            image_results_for_layer = []
            for i in range(n_imgs):
                feats = features[key][i][layer]
                result = do_linear_probe(feats, ramp, probe_by_channel=True, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
                image_results_for_layer.append(result)
            averaged = average_results(image_results_for_layer)
            model_to_layer_results[key].append(averaged)
            

In [8]:
if not LOAD_RESULTS:
    model_to_layer_channel_scores: dict[ModelTypes, np.ndarray] = {key: None for key in selected_models}
    for key in selected_models:
        layer_channel_scores = np.array([res[0] for res in model_to_layer_results[key]])
        model_to_layer_channel_scores[key] = layer_channel_scores

In [9]:

def dump_scores_to_file(filename: str, model_to_layer_channel_scores: dict):
    with open(filename, "wb") as f:
        pdump(model_to_layer_channel_scores, f)


try:
    mkdir("data/pos_fingerprint")
except FileExistsError:
    pass

STORE_RESULTS = False
if STORE_RESULTS:
    dump_scores_to_file(f"{DATA_DIR}/scores.pkl", model_to_layer_channel_scores)

In [10]:
# print(model_to_layer_results['alibi_coco_dinov2_s'].shape)

In [11]:
from matplotlib.collections import PatchCollection
from matplotlib.patches import Rectangle


def add_red_square_overlay(ax: plt.Axes, mask: np.ndarray, sf_h: float, sf_w: int) -> None:
    patches: list[Rectangle] = []
    y_inds, x_inds  = np.nonzero(mask)
    for x, y in zip(x_inds, y_inds):
        rect = Rectangle((x - sf_w / 2, y - sf_h / 2), sf_w, sf_h)
        patches.append(rect)
    pc = PatchCollection(patches, color='red', facecolor='none', lw=0.25)
    ax.add_collection(pc)

In [12]:
print(plt.rcParams["font.size"])
print(plt.rcParams["axes.labelsize"])
print(plt.rcParams["xtick.labelsize"])
print(plt.rcParams["ytick.labelsize"])

10.0
medium
medium
medium


In [25]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec


W, H = 3, 2.75  * 2.3
fig = plt.figure(figsize=(W, H))

plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False
add_custom_font('resources/fonts', 'Grotesk')

n_cols = len(selected_models)
n_rows = 3

W, H = 2.25, 4.5  # Reduced overall figure height scaling
w_spacing = [1.0 for _ in range(n_cols)]

# 0: Ramp (snug), 1: Colorbar strip, 2: Heatmaps
h_spacing = [0.24, 0.06, 1.2]

# fig = plt.figure(figsize=(W * n_cols * 0.45, H * sum(h_spacing)))

# Tight hspace (0.18 removes the dead padding between rows)
gs = GridSpec(
    n_rows,
    n_cols,
    figure=fig,
    width_ratios=w_spacing,
    height_ratios=h_spacing,
    hspace=0.28,
    wspace=0.3,
)

mid_start = (n_cols - 2) // 2
mid_end = mid_start + 2

h, w = 34, 34
ramp_arr = get_ramp(ramp, h, w)

# 1. Ramp
ramp_ax = fig.add_subplot(gs[0, mid_start:mid_end])
ramp_ax.imshow(
    ramp_arr,
    aspect="equal",
    vmin=0,
    vmax=1,
    interpolation="nearest",
    cmap="viridis",
)
ramp_ax.set_xticks([])
ramp_ax.set_yticks([])
ramp_ax.set_title("Target ramp", fontsize=7, pad=3)

mask = gen_sample_mask(
    (h, w), ramp, STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK
)
add_red_square_overlay(ramp_ax, mask, 1, 1)

# 2. Horizontal colorbar (row 1)
cbar_ax = fig.add_subplot(gs[1, :])

# 3. Heatmaps (row 2)
fingerprint_axes = []
last_im = None
for i, key in enumerate(selected_models):
    ax = fig.add_subplot(gs[2, i])
    fingerprint_axes.append(ax)
    last_im = ax.imshow(
        model_to_layer_channel_scores[key].T,
        aspect="auto",
        vmin=0,
        vmax=0.5,
        cmap="viridis",
        interpolation="nearest",
    )
    name = model_names[key].replace("(COCO)", "")
    weight = 700 if "alibi" in key else 500
    ax.set_title(name, weight=weight, fontsize=7)
    ax.tick_params(labelsize=6, width=1.5)
    ax.set_xticks([0, 5, 11])
    ax.set_xticklabels([1, 6, 12])

    if i > 0:
        ax.set_yticks([])
    else:
        ax.set_ylabel("Channel")

# Populate colorbar now that last_im is defined
cbar = fig.colorbar(last_im, cax=cbar_ax, orientation="horizontal")
cbar.ax.tick_params(labelsize=6)
cbar_ax.set_title(r"Per-channel $R^2$ scores", fontsize=7, pad=3)

# Center 'Layer' between the middle two axes
fig.canvas.draw()
left_ax_pos = fingerprint_axes[mid_start].get_position()
right_ax_pos = fingerprint_axes[mid_end - 1].get_position()
center_x = (left_ax_pos.x0 + right_ax_pos.x1) / 2.0
label_y = left_ax_pos.y0 - 0.04

fig.text(center_x, label_y, "Layer", ha="center", va="top", fontsize=7)

SAVE = True
if SAVE:
    plt.savefig("saved/03.pdf", dpi=300, bbox_inches="tight")
    plt.close()

findfont: Failed to find font weight normal for Host Grotesk, now using 300.
findfont: Failed to find font weight normal for Host Grotesk, now using 300.
findfont: Failed to find font weight normal for Host Grotesk, now using 300.
findfont: Failed to find font weight normal for Host Grotesk, now using 300.
findfont: Failed to find font weight 500 for Host Grotesk, now using 300.
